# RectangleBuilderGreaterArea

[Codility task](https://app.codility.com/programmers/trainings/2/rectangle_builder_greater_area/) - Exercise 2, 2016 Contest

A rectangular pen needs four pieces of fence: two of length `a` and two of length `b`. `solution(A, X)` counts the **different** pens (unordered pairs `{a, b}`) with area `a * b >= X` that can be built from the pieces `A`, or returns `-1` if the count exceeds `1,000,000,000`.

| A | X | result |
|---|---|--------|
| `[1, 2, 5, 1, 1, 2, 3, 5, 1]` | 5 | 2 (`1x5` and `2x5`) |

Assumptions: `N` in `[0..100,000]`, `X` and the lengths in `[1..10^9]`.

## Idea

- `a != b` needs at least **two** pieces of each length; `a == b` (a square) needs at least **four** pieces of that length.
- Count the pieces per length and keep the sorted list `lengths` of the distinct lengths with at least two pieces.
- **Squares**: lengths with at least four pieces and `a * a >= X`.
- **Rectangles** `a < b`: for each `a` the valid `b` are those with `b >= ceil(X / a)` located after `a` in the sorted list, a suffix found with `bisect_left`.

The count can reach about `1.25 * 10^9` (50,000 lengths with two pieces each), so stop as soon as it exceeds the limit. Time O(N log N).

In [ ]:
from bisect import bisect_left
from collections import Counter

LIMIT = 1_000_000_000


def solution(A, X):
    counts = Counter(A)
    lengths = sorted(value for value, c in counts.items() if c >= 2)
    total = sum(1 for value, c in counts.items() if c >= 4 and value * value >= X)     # squares
    for i, a in enumerate(lengths):
        needed = -(-X // a)                                                            # ceil(X / a)
        total += len(lengths) - bisect_left(lengths, needed, i + 1)                    # partners b > a with b >= needed
        if total > LIMIT:
            return -1
    return -1 if total > LIMIT else total

## Official example

In [ ]:
assert solution([1, 2, 5, 1, 1, 2, 3, 5, 1], 5) == 2
print("example OK")

## Edge cases

In [ ]:
assert solution([], 1) == 0                                     # no pieces at all
assert solution([3], 1) == 0
assert solution([2, 2], 1) == 0                                 # two pieces are not enough for a pen
assert solution([2, 2, 2], 1) == 0
assert solution([2, 2, 2, 2], 4) == 1                           # the square 2x2
assert solution([2, 2, 2, 2], 5) == 0                           # too small
assert solution([1, 1, 2, 2], 2) == 1                           # 1x2
assert solution([1, 1, 2, 2], 3) == 0
assert solution([1, 1, 1, 1, 2, 2, 2, 2], 2) == 2               # 1x2 and 2x2 (1x1 is too small)
assert solution([1, 1, 1, 1, 2, 2, 2, 2], 1) == 3               # 1x1, 1x2, 2x2
assert solution([10**9] * 4, 10**9) == 1                        # 10^18 is far above X
assert solution([10**9] * 2 + [1] * 2, 10**9) == 1              # exactly X
assert solution([10**9] * 2 + [1] * 2, 10**9 + 1) == 0          # one too small
print("edge cases OK")

## The `-1` limit

With every length `1..n` present exactly twice there are no squares and every pair of lengths gives a pen (for `X = 1`), so the answer is `n (n - 1) / 2`. It crosses `10^9` between `n = 44,721` (999,961,560) and `n = 44,722` (1,000,006,281).

In [ ]:
def all_pairs(n):
    return [v for v in range(1, n + 1) for _ in range(2)]


assert solution(all_pairs(44_721), 1) == 44_721 * 44_720 // 2 == 999_961_560
assert solution(all_pairs(44_722), 1) == -1
assert solution(all_pairs(50_000), 1) == -1
print("limit OK")

## Brute force

The reference literally tries **every set of four pieces**: sorted, they must look like `[a, a, b, b]`; the pen `{a, b}` is added to a set when its area is large enough. Distinct pens are the elements of that set.

In [ ]:
import itertools
import random


def brute(A, X):
    pens = set()
    for quad in itertools.combinations(A, 4):
        w, x, y, z = sorted(quad)
        if w == x and y == z and w * y >= X:
            pens.add((w, y))
    return len(pens)


rng = random.Random(6)
for _ in range(3000):
    n = rng.randint(0, 11)
    A = [rng.randint(1, 5) for _ in range(n)]
    X = rng.randint(1, 30)
    assert solution(A, X) == brute(A, X), (A, X)
print("matches brute force on 3000 random inputs")

## Performance

In [ ]:
import time

N = 100_000
cases = {
    "random, many repeats": ([rng.randint(1, 1000) for _ in range(N)], 5000),
    "random, wide values": ([rng.randint(1, 10**9) for _ in range(N)], 10**9),
    "all pairs (-1 expected)": (all_pairs(50_000), 1),
    "all equal": ([7] * N, 49),
    "x huge": ([rng.randint(1, 10**9) for _ in range(N)], 10**9),
}
for name, (A, X) in cases.items():
    start = time.perf_counter()
    result = solution(A, X)
    print(f"{name:26s} -> {result:>10} in {time.perf_counter() - start:.3f}s")

## Notes

- Think in terms of **multisets of side lengths**: the problem is about distinct unordered pairs, not about which physical pieces are used.
- Squares are the only case where one length is used twice as a side pair, hence "at least four".
- `ceil(X / a)` without floats: `-(-X // a)`.
- Sorting plus `bisect` turns a double loop into O(n log n).